# E2: 2D CNN (Folded Ladder) for DNA Thermodynamics (Standardized)

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E2  
**Thesis Chapter:** Chapter 3 — Spatial Bias via Folded Ladder Encoding  

## Core Idea
DNA hairpin sequences are computationally **folded into a 2D 'ladder'** image, where:
- Row 0 = 5′ strand (top)
- Row 1 = H-bond indicator + backbone turn
- Row 2 = 3′ strand reversed (bottom)

This encoding captures **spatial co-location of Watson-Crick base pairs**. A 2D CNN with learned attention pooling can then detect both local stacking motifs (small kernels) and stem-level patterns (larger kernels).

**Input:** `(6, 3, 15)` — 6 channels (4 one-hot nt + 1 H-bond + 1 backbone), height=3, width=max_width.  
**Inductive bias:** 2D spatial locality + learned attention pooling weights positions by thermodynamic importance.

In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb
import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

COLORS = {'E0_GNN':'#7f8c8d','E1_1DCNN':'#3498db','E2_2DCNN':'#e74c3c',
          'E3_SAT':'#9b59b6','E4_PINN':'#e67e22','E5_Hybrid':'#1abc9c'}
MODEL_COLOR = COLORS['E2_2DCNN']

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
MAX_WIDTH = 15   # ceil(24/2)+1 — covers all arr sequences (11–24 nt)

DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'

config = dict(
    model_name     = '2D_CNN_FoldedLadder',
    experiment_id  = 'E2',
    in_channels    = 6,
    dropout        = 0.2,
    n_epoch        = 200,
    batch_size     = 256,
    lr             = 1e-3,
    weight_decay   = 1e-5,
    grad_clip      = 1.0,
    dataset        = 'arr',
    wandb_project  = 'NNN_Thesis_Experiments',
    checkpoint_dir = 'MyExperiments/2DCNN/models',
)
print('Config:', config)

Config: {'model_name': '2D_CNN_FoldedLadder', 'experiment_id': 'E2', 'in_channels': 6, 'dropout': 0.2, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/2DCNN/models'}


In [3]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)
with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH','Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH','Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH','Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

sumstats = {
    'dH_min': float(train_df['dH'].min()), 'dH_max': float(train_df['dH'].max()),
    'Tm_min': float(train_df['Tm'].min()), 'Tm_max': float(train_df['Tm'].max()),
}
def normalize(v, mn, mx):   return (v - mn) / (mx - mn)
def unnormalize(v, mn, mx): return v * (mx - mn) + mn

print(f'Train {len(train_df):,}  Val {len(val_df):,}  Test {len(test_df):,}  (arr only)')
print(f'dH [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}]  Tm [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}]')

Train 25,025  Val 1,318  Test 1,387  (arr only)
dH [-68.2, -2.7]  Tm [13.6, 68.6]


In [4]:
# ── 4. Encoding: 2D Folded Ladder (6, 3, max_width) ───────────────────────────
NT_MAP = {'A':0,'T':1,'G':2,'C':3}

def encode_2d_hairpin(seq, struct, max_width=MAX_WIDTH):
    n_stem   = struct.count('(')
    n_loop   = struct.count('.')
    half_loop = n_loop // 2
    has_mid  = (n_loop % 2 == 1)
    fold_len = n_stem + half_loop
    top_seq  = seq[:fold_len]
    if has_mid:
        mid_nt = seq[fold_len]; bot_seq = seq[fold_len+1:][::-1]
    else:
        mid_nt = None;           bot_seq = seq[fold_len:][::-1]
    hbond  = [1.]*n_stem + [0.]*half_loop
    tensor = np.zeros((6,3,max_width), dtype=np.float32)
    for i,nt in enumerate(top_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()],0,i] = 1.
    for i,nt in enumerate(bot_seq):
        if nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()],2,i] = 1.
    for i,h in enumerate(hbond):  tensor[4,1,i] = h
    bb_col = min(fold_len, max_width-1)
    tensor[5,0,bb_col] = tensor[5,1,bb_col] = tensor[5,2,bb_col] = 1.
    if has_mid and mid_nt and mid_nt.upper() in NT_MAP: tensor[NT_MAP[mid_nt.upper()],1,bb_col] = 1.
    return tensor

def encode_2d_duplex(s1, s2, max_width=MAX_WIDTH):
    tensor = np.zeros((6,3,max_width), dtype=np.float32)
    for i,nt in enumerate(s1):
        if i<max_width and nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()],0,i] = 1.
    for i,nt in enumerate(s2[::-1]):
        if i<max_width and nt.upper() in NT_MAP: tensor[NT_MAP[nt.upper()],2,i] = 1.
    for i in range(min(len(s1),max_width)): tensor[4,1,i] = 1.
    return tensor

def encode_row_2d(row, max_width=MAX_WIDTH):
    refseq = str(row['RefSeq'])
    struct = str(row['TargetStruct'])
    if '[' in refseq:
        try: refseq_list = eval(refseq)
        except: refseq_list = [refseq]
    elif isinstance(row['RefSeq'], list): refseq_list = row['RefSeq']
    else: refseq_list = None
    if '+' in struct:
        if refseq_list and len(refseq_list)==2: arr = encode_2d_duplex(refseq_list[0], refseq_list[1], max_width)
        else:
            plus = struct.index('+')
            arr = encode_2d_duplex(refseq[:plus], refseq[plus+1:], max_width)
    else:
        seq = ''.join(refseq_list) if refseq_list else refseq
        arr = encode_2d_hairpin(seq, struct, max_width)
    return torch.tensor(arr, dtype=torch.float)

_r = df.iloc[0]
print(f'Encoding shape: {encode_row_2d(_r).shape}  (6 ch × 3 rows × {MAX_WIDTH} width)')

Encoding shape: torch.Size([6, 3, 15])  (6 ch × 3 rows × 15 width)


In [5]:
# ── 5. Dataset & DataLoaders ──────────────────────────────────────────────────

class NNN2DDataset(Dataset):
    def __init__(self, df, sumstats):
        self.df = df.reset_index(drop=False); self.ss = sumstats
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = encode_row_2d(row)
        y = torch.tensor([normalize(row['dH'], self.ss['dH_min'], self.ss['dH_max']),
                          normalize(row['Tm'], self.ss['Tm_min'], self.ss['Tm_max'])], dtype=torch.float)
        return x, y

train_ds = NNN2DDataset(train_df, sumstats)
val_ds   = NNN2DDataset(val_df,   sumstats)
test_ds  = NNN2DDataset(test_df,  sumstats)
train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512,                  shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512,                  shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}')
_x,_y = next(iter(train_loader)); print(f'Batch — x: {_x.shape}  y: {_y.shape}')

Train batches: 98
Batch — x: torch.Size([256, 6, 3, 15])  y: torch.Size([256, 2])


In [6]:
# ── 6. Model: 2D CNN with Learned Attention Pooling ───────────────────────────

class AttentionPool2d(nn.Module):
    """
    Learns per-position attention weights over (H, W) spatial dimensions.
    score = v^T · tanh(W · h)    then softmax → weighted sum.
    """
    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, 64, 1), nn.Tanh(),
            nn.Conv2d(64, 1, 1),
        )
    def forward(self, x):
        B,C,H,W = x.shape
        w = torch.softmax(self.attn(x).view(B,1,-1), dim=-1)  # (B,1,H*W)
        return (x.view(B,C,-1) * w).sum(-1)                    # (B,C)


class DNA_2DCNN(nn.Module):
    """
    3-block 2D CNN on the folded ladder encoding.
    Kernels: (3,3) → (3,3) → (3,5) capture from single base-pair up to 5-nt stacking context.
    """
    def __init__(self, in_channels=6, dropout=0.2):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels,64,(3,3),padding=(1,1)), nn.BatchNorm2d(64),  nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(64,128,(3,3),padding=(1,1)),          nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
            nn.Conv2d(128,128,(3,5),padding=(1,2)),         nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(dropout),
        )
        self.pool = AttentionPool2d(128)
        self.head = nn.Sequential(
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64,2),
        )
    def forward(self, x):
        return self.head(self.pool(self.conv_block(x)))


model = DNA_2DCNN(in_channels=config['in_channels'], dropout=config['dropout']).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: DNA_2DCNN  |  Parameters: {n_params:,}')

Model: DNA_2DCNN  |  Parameters: 340,611


In [7]:
# ── 7. Metrics & Evaluation Helpers ──────────────────────────────────────────

def compute_metrics(pred_norm, true_norm, sumstats):
    if torch.is_tensor(pred_norm): pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm): true_norm = true_norm.cpu().numpy()
    dH_p = pred_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_p = pred_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dH_t = true_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_t = true_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dG_p = dH_p*(1.-(273.15+37.)/(273.15+Tm_p))
    dG_t = dH_t*(1.-(273.15+37.)/(273.15+Tm_t))
    metrics = {}
    for tag,p,t in [('dH',dH_p,dH_t),('Tm',Tm_p,Tm_t),('dG_37',dG_p,dG_t)]:
        mask = np.isfinite(t)&np.isfinite(p)
        if mask.sum()<2:
            metrics[f'{tag}_mae']=metrics[f'{tag}_rmse']=metrics[f'{tag}_r2']=float('nan')
        else:
            d=p[mask]-t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(d)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(d**2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask],p[mask]))
    return metrics, dH_p, Tm_p, dH_t, Tm_t

@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    model.eval(); preds,trues=[],[]
    for x,y in loader:
        preds.append(model(x.to(device)).cpu()); trues.append(y)
    return compute_metrics(torch.cat(preds), torch.cat(trues), sumstats)

print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['n_epoch'], eta_min=1e-5)

history = {'train_loss':[], 'val_dH_mae':[], 'val_Tm_mae':[], 'val_dG_mae':[], 'val_dH_rmse':[], 'val_Tm_rmse':[]}
os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

_run_name = 'E2_2DCNN_FoldedLadder_AttentionPool'
_kw = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_mode = os.environ.get('WANDB_MODE','').strip().lower()
if _mode in ('offline','disabled'):
    run = wandb.init(mode=_mode, **_kw)
else:
    try:    run = wandb.init(**_kw)
    except Exception as e:
        print(f'WandB online failed ({e}), offline.'); run = wandb.init(mode='offline', **_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf'); start = time.time()

for epoch in range(config['n_epoch']):
    model.train(); train_loss = 0.
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    vm, *_ = evaluate(model, val_loader, sumstats, device)
    history['train_loss'].append(train_loss)
    history['val_dH_mae'].append(vm['dH_mae'])
    history['val_Tm_mae'].append(vm['Tm_mae'])
    history['val_dG_mae'].append(vm['dG_37_mae'])
    history['val_dH_rmse'].append(vm['dH_rmse'])
    history['val_Tm_rmse'].append(vm['Tm_rmse'])

    wandb.log({'epoch':epoch,'train_loss':train_loss,**{f'val_{k}':v for k,v in vm.items()},'lr':scheduler.get_last_lr()[0]})

    if vm['dG_37_mae'] < best_val_dG:
        best_val_dG = vm['dG_37_mae']
        torch.save(model.state_dict(), os.path.join(config['checkpoint_dir'],'best_cnn2d_model.pt'))

    if (epoch+1) % 20 == 0:
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} | loss {train_loss:.4f} | dH {vm['dH_mae']:.3f} | Tm {vm['Tm_mae']:.3f} | dG {vm['dG_37_mae']:.3f} | {(time.time()-start)/60:.1f}min")

run.finish()
with open('out/cnn2d_history.json','w') as f: json.dump(history, f)
print(f'Done. Best val dG MAE: {best_val_dG:.4f}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.
wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: E2_2DCNN_FoldedLadder_AttentionPool  |  mode: online
Ep  20/200 | loss 0.0068 | dH 3.566 | Tm 2.443 | dG 0.211 | 2.2min
Ep  40/200 | loss 0.0047 | dH 3.025 | Tm 2.304 | dG 0.197 | 4.3min
Ep  60/200 | loss 0.0040 | dH 3.050 | Tm 2.016 | dG 0.179 | 6.3min
Ep  80/200 | loss 0.0036 | dH 2.922 | Tm 1.839 | dG 0.171 | 8.4min
Ep 100/200 | loss 0.0034 | dH 2.991 | Tm 1.776 | dG 0.171 | 10.5min
Ep 120/200 | loss 0.0031 | dH 2.916 | Tm 1.692 | dG 0.167 | 12.5min
Ep 140/200 | loss 0.0028 | dH 2.883 | Tm 1.654 | dG 0.163 | 14.6min
Ep 160/200 | loss 0.0027 | dH 2.839 | Tm 1.609 | dG 0.159 | 16.7min
Ep 180/200 | loss 0.0025 | dH 2.869 | Tm 1.575 | dG 0.158 | 19.6min
Ep 200/200 | loss 0.0024 | dH 2.877 | Tm 1.565 | dG 0.158 | 21.7min


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
lr,██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▇▇▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,█▆▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_Tm_r2,▃▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇███████████████████████
val_Tm_rmse,█▄▃▃▂▂▂▂▂▂▂▂▁▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dG_37_mae,█▃▄▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dG_37_r2,▁▃▆▆▆▇▇▇▇▇▇█▇█▇█████████████████████████
val_dG_37_rmse,█▆▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dH_mae,▇▆█▄█▄▃▄▄▃▂▂▃▄▂▂▃▁▂▂▁▂▂▂▂▂▁▁▂▁▂▁▁▁▁▁▁▁▁▁
+2,...


Done. Best val dG MAE: 0.1561


In [9]:
# ── 9. Final Evaluation ───────────────────────────────────────────────────────

model.load_state_dict(torch.load(os.path.join(config['checkpoint_dir'],'best_cnn2d_model.pt'), map_location=device))
val_m,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_m, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Val (arr) ===');  [print(f'  {t}  MAE {val_m[f"{k}_mae"]:.3f}  R2 {val_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]
print('=== Test (arr) ==='); [print(f'  {t}  MAE {test_m[f"{k}_mae"]:.3f}  R2 {test_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]

ev = pd.DataFrame({'dH_pred':dH_vp,'dH_true':dH_vt,'Tm_pred':Tm_vp,'Tm_true':Tm_vt})
ev['dG_pred'] = ev['dH_pred']*(1-310.15/(273.15+ev['Tm_pred']))
ev['dG_true'] = ev['dH_true']*(1-310.15/(273.15+ev['Tm_true']))
ev.to_csv('out/cnn2d_val_eval.csv', index=False)

run_log = dict(experiment_id='E2', model='DNA_2DCNN', config=config,
               n_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
               val_metrics=val_m, test_metrics=test_m,
               best_checkpoint=os.path.join(config['checkpoint_dir'],'best_cnn2d_model.pt'))
with open('out/cnn2d_run_log.json','w') as f: json.dump(run_log, f, indent=2)
print('Saved: out/cnn2d_val_eval.csv  out/cnn2d_run_log.json')

=== Val (arr) ===
  dH  MAE 2.846  R2 0.874
  Tm  MAE 1.568  R2 0.957
  dG37  MAE 0.156  R2 0.952
=== Test (arr) ===
  dH  MAE 2.830  R2 0.877
  Tm  MAE 1.621  R2 0.951
  dG37  MAE 0.159  R2 0.952
Saved: out/cnn2d_val_eval.csv  out/cnn2d_run_log.json


In [13]:
# ── 10. Convergence Curves (F2 contribution) ──────────────────────────────────
os.makedirs('out/figures', exist_ok=True)
ep = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1,3,figsize=(14,4),facecolor='#f8f9fa')
for ax,(k,yl) in zip(axes,[('val_dH_mae','Val dH MAE (kcal/mol)'),('val_Tm_mae','Val Tm MAE (deg C)'),('val_dG_mae','Val dG37 MAE (kcal/mol)')]):
    ax.plot(ep,history[k],color=MODEL_COLOR,lw=2,label='E2: 2D CNN')
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl); ax.legend(fontsize=9); sns.despine(ax=ax)
fig.suptitle('E2: 2D CNN (Folded Ladder) Convergence',fontsize=11,fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/cnn2d_convergence.png',dpi=300,bbox_inches='tight')
plt.show(); print('Saved: out/figures/cnn2d_convergence.png')

Saved: out/figures/cnn2d_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_31208\3218321125.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/cnn2d_convergence.png')


In [14]:
# ── 11. Scatter Plots (F3 contribution) ───────────────────────────────────────
AXIS_LIMITS = {'dH':(-55,-5),'Tm':(20,60),'dG_37':(-7,5)}
dG_vp = dH_vp*(1-310.15/(273.15+Tm_vp)); dG_vt = dH_vt*(1-310.15/(273.15+Tm_vt))
fig,axes = plt.subplots(1,3,figsize=(14,5),facecolor='#f8f9fa')
for ax,(p,t,tag,unit) in zip(axes,[(dH_vp,dH_vt,'dH','kcal/mol'),(Tm_vp,Tm_vt,'Tm','deg C'),(dG_vp,dG_vt,'dG_37','kcal/mol')]):
    lim=AXIS_LIMITS[tag]; m=np.isfinite(p)&np.isfinite(t)
    ax.scatter(t[m],p[m],s=4,alpha=0.4,color=MODEL_COLOR,rasterized=True)
    ax.plot(lim,lim,'k--',alpha=0.3,lw=1.5)
    mae=np.mean(np.abs(p[m]-t[m])); r2=r2_score(t[m],p[m])
    ax.text(0.05,0.93,f'MAE={mae:.3f}\nR2={r2:.3f}',transform=ax.transAxes,fontsize=8.5,va='top',bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel(f'Measured {tag}'); ax.set_ylabel(f'Predicted {tag}'); ax.set_title(tag,fontweight='bold'); sns.despine(ax=ax)
fig.suptitle('E2: 2D CNN Predicted vs Measured (Val)',fontsize=11)
plt.tight_layout()
plt.savefig('out/figures/cnn2d_scatter.png',dpi=300,bbox_inches='tight')
plt.show(); print('Saved: out/figures/cnn2d_scatter.png')

Saved: out/figures/cnn2d_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_31208\562555120.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/cnn2d_scatter.png')


In [15]:
# ── 12. lit_uv Generalization (F4 contribution) ───────────────────────────────
lit_df = df[df['dataset']=='lit_uv'].copy()
print(f'lit_uv: {len(lit_df)} sequences (Tm-only)')

model.eval(); Tm_preds = []
with torch.no_grad():
    for _, row in lit_df.iterrows():
        x = encode_row_2d(row).unsqueeze(0).to(device)
        out = model(x)
        Tm_preds.append(unnormalize(out[0,1].item(), sumstats['Tm_min'], sumstats['Tm_max']))

Tm_true = lit_df['Tm'].values
lit_mae = float(np.mean(np.abs(np.array(Tm_preds)-Tm_true)))
print(f'lit_uv Tm MAE: {lit_mae:.3f} degC')

with open('out/cnn2d_run_log.json') as f: rl = json.load(f)
rl['lit_uv_Tm_mae'] = lit_mae
with open('out/cnn2d_run_log.json','w') as f: json.dump(rl, f, indent=2)
print('Updated cnn2d_run_log.json with lit_uv result.')

lit_uv: 348 sequences (Tm-only)
lit_uv Tm MAE: 9.945 degC
Updated cnn2d_run_log.json with lit_uv result.
